In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# IBM Circuit function

> **Note:** * Qiskit Functions to funkcja eksperymentalna dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan i On-Prem (przez IBM Quantum Platform API). Są one w fazie podglądu i mogą ulec zmianie.

## Przegląd
IBM&reg; Circuit function przyjmuje [abstrakcyjne PUBs](./primitive-input-output) jako dane wejściowe i zwraca złagodzone wartości oczekiwane jako dane wyjściowe. Ta Circuit function obejmuje zautomatyzowany i dostosowany potok, który pozwala badaczom skupić się na odkrywaniu algorytmów i zastosowań.
## Opis
Po przesłaniu swojego PUB, twoje abstrakcyjne Circuit i obserwowalności są automatycznie transpilowane, wykonywane na sprzęcie i poddawane post-processingu w celu zwrócenia złagodzonych wartości oczekiwanych. W tym celu łączy następujące narzędzia:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), w tym automatyczny dobór opartych na AI i heurystycznych przejść Transpiler do tłumaczenia abstrakcyjnych Circuit na zoptymalizowane pod kątem sprzętu Circuit ISA
- [Tłumienie i łagodzenie błędów wymagane do obliczeń na skalę użytkową](./error-mitigation-and-suppression-techniques), w tym skręcanie pomiarów i bramek, dynamiczne rozłączanie, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE) i Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), do wykonywania PUBs ISA na sprzęcie i zwracania złagodzonych wartości oczekiwanych

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)
## Pierwsze kroki
Uwierzytelnij się przy użyciu swojego [klucza API](http://quantum.cloud.ibm.com/) i wybierz Qiskit Function w następujący sposób. (Ten fragment kodu zakłada, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w swoim lokalnym środowisku.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Przykład
 Aby zacząć, wypróbuj ten podstawowy przykład:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Sprawdź [status](/guides/functions#check-job-status) swojego zadania Qiskit Function lub pobierz [wyniki](/guides/functions#retrieve-results) w następujący sposób:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Wyniki mają taki sam format jak [wynik Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Dane wejściowe
Poniższa tabela zawiera wszystkie parametry wejściowe akceptowane przez IBM Circuit function. Kolejna sekcja [Opcje](#options) zawiera więcej szczegółów na temat dostępnych `options`.
| Nazwa      | Typ                       | Opis                                                                                                                                                                                                                         | Wymagany | Domyślnie                                                                  | Przykład                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Nazwa Backend, do którego ma być skierowane zapytanie.                                                                                                                                                                                              | Tak      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Iterowalny zbiór abstrakcyjnych obiektów PUB-like (primitive unified bloc), takich jak krotki `(circuit, observables)` lub `(circuit, observables, parameter_values)`. Więcej informacji znajdziesz w [Przeglądzie PUBs](/guides/primitive-input-output#overview-of-pubs). Circuit mogą być abstrakcyjne (non-ISA). | Tak      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Opcje wejściowe. Więcej szczegółów znajdziesz w sekcji **Opcje**.                                                                                                                                                                                | Nie       | Szczegóły w sekcji **Opcje**.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Nazwa zasobu chmury instancji do użycia w tym formacie.                                                                                                                                                                                        | Nie       | Jedna jest losowo wybierana, jeśli twoje konto ma dostęp do wielu instancji. | `CRN`                   |
### Opcje
#### Struktura
Podobnie jak w przypadku prymitywów Qiskit Runtime, opcje dla IBM Circuit function można określić jako zagnieżdżony słownik. Powszechnie używane opcje, takie jak ``optimization_level`` i ``mitigation_level``, znajdują się na pierwszym poziomie. Inne, bardziej zaawansowane opcje są pogrupowane w różne kategorie, takie jak ``resilience``.

#### Wartości domyślne
Jeśli nie określisz wartości dla opcji, używana jest wartość domyślna określona przez usługę.

#### Poziom mitigacji
IBM Circuit function obsługuje również `mitigation_level`. Poziom mitigacji określa, ile tłumienia i łagodzenia błędów zastosować do zadania. Wyższe poziomy generują dokładniejsze wyniki kosztem dłuższego czasu przetwarzania. Stopień redukcji błędów zależy od zastosowanej metody. Poziom mitigacji abstrahuje szczegółowy dobór metod łagodzenia i tłumienia błędów, aby umożliwić użytkownikom rozumowanie o kompromisie między kosztem a dokładnością odpowiednim dla ich zastosowania. Poniższa tabela pokazuje odpowiednie metody dla każdego poziomu.

> **Note:** Mimo że nazwy są podobne, `mitigation_level` stosuje inne techniki niż te używane przez `resilience_level` Estimator.

Podobnie jak ``resilience_level`` w Qiskit Runtime Estimator, ``mitigation_level`` określa podstawowy zestaw starannie dobranych opcji. Wszelkie opcje ręcznie określone oprócz poziomu mitigacji są stosowane na wierzchu podstawowego zestawu opcji zdefiniowanych przez poziom mitigacji. Dlatego w zasadzie można ustawić poziom mitigacji na 1, ale wyłączyć mitigację pomiarów, choć nie jest to zalecane.

| **Poziom mitigacji** | **Technika** |
|:-:|:-:|
| 1 [Domyślny] | Dynamical decoupling + skręcanie pomiarów + TREX  |
| 2 | Poziom 1 + skręcanie bramek + ZNE przez fałdowanie bramek |
| 3 | Poziom 1 + skręcanie bramek + ZNE przez PEA |

Poniższy przykład demonstruje ustawianie poziomu mitigacji:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Wszystkie dostępne opcje
Oprócz `mitigation_level`, IBM Circuit function udostępnia również wybraną liczbę zaawansowanych opcji, które pozwalają dostroić kompromis między kosztem a dokładnością. Poniższa tabela pokazuje wszystkie dostępne opcje:

| Opcja | Podopcja | Pod-podopcja | Opis | Wybory | Domyślnie |
|-|-|-|-|-|-|
| default_precision |  |  | Domyślna precyzja do użycia dla dowolnego PUB lub wywołania `run()`,<br/>które jej nie określa.<br/>Każdy wejściowy PUB może określić własną precyzję. Jeśli metoda `run()` otrzyma precyzję, ta wartość jest używana dla wszystkich PUBs w wywołaniu `run()`, które nie określają własnej.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maksymalny czas wykonania w sekundach, oparty<br/>na użyciu QPU (nie czasie zegarowym). Użycie QPU to<br/>czas, przez który QPU jest dedykowany do przetwarzania twojego zadania. Jeśli zadanie przekroczy ten limit czasu, zostanie przymusowo anulowane. | Liczba całkowita sekund w zakresie [1, 10800] |  |
| mitigation_level |  |  | Ile tłumienia i łagodzenia błędów zastosować. Więcej informacji o metodach używanych na każdym poziomie znajdziesz w sekcji [Poziom mitigacji](#mitigation-level). | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Ile optymalizacji przeprowadzić na Circuit. [Wyższe poziomy](/guides/set-optimization) generują bardziej zoptymalizowane Circuit kosztem dłuższego czasu transpilacji. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Czy włączyć dynamical decoupling. Opis metody znajdziesz w [Technikach tłumienia i łagodzenia błędów](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling).  | True/False | True |
|  | sequence_type |  | Której sekwencji dynamical decoupling użyć.<br/>* `XX`: użyj sekwencji `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: użyj sekwencji `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: użyj sekwencji<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Czy stosować skręcanie Gate 2-Qubitowych Clifford. | True/False | False |
|  | enable_measure |  | Czy włączyć skręcanie pomiarów. | True/False | True |
| resilience | measure_mitigation |  | Czy włączyć metodę mitigacji błędów pomiarowych TREX. Opis metody znajdziesz w [Technikach tłumienia i łagodzenia błędów](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex).  | True/False | True |
|  | zne_mitigation |  | Czy włączyć metodę mitigacji błędów Zero Noise Extrapolation. Opis metody znajdziesz w [Technikach tłumienia i łagodzenia błędów](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne).  | True/False | False |
|  | zne | amplifier | Której techniki użyć do wzmacniania szumu. Jedna z: <br/> - `gate_folding` (domyślna) używa fałdowania Gate 2-Qubitowych do wzmacniania szumu. Jeśli współczynnik szumu wymaga wzmocnienia tylko podzbioru Gate, są one wybierane losowo.<br/><br/> - `gate_folding_front` używa fałdowania Gate 2-Qubitowych do wzmacniania szumu. Jeśli współczynnik szumu wymaga wzmocnienia tylko podzbioru Gate, są one wybierane z przodu topologicznie uporządkowanego obwodu DAG.<br/><br/> - `gate_folding_back` używa fałdowania Gate 2-Qubitowych do wzmacniania szumu. Jeśli współczynnik szumu wymaga wzmocnienia tylko podzbioru Gate, są one wybierane z tyłu topologicznie uporządkowanego obwodu DAG.<br/><br/> - `pea` używa techniki zwanej Probabilistic error amplification (PEA) do wzmacniania szumu. Opis metody znajdziesz w [Technikach tłumienia i łagodzenia błędów](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea).  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Współczynniki szumu do użycia przy wzmacnianiu szumu. | lista liczb zmiennoprzecinkowych; każda >= 1 | (1, 1.5, 2) dla PEA i (1, 3, 5) w pozostałych przypadkach. |
|  |  | extrapolator | Współczynniki szumu, w których oceniać modele ekstrapolacji dopasowania. Ta opcja nie ma wpływu na wykonanie ani dopasowanie modelu w żaden sposób; określa jedynie punkty, w których obiekty `extrapolator` są oceniane i zwracane w polach danych o nazwach `evs_extrapolated` i `stds_extrapolated`. | jeden lub więcej z `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Czy włączyć metodę mitigacji błędów Probabilistic Error Cancellation. Opis metody znajdziesz w [Technikach tłumienia i łagodzenia błędów](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec).  | True/False | False |
|  | pec | max_overhead | Maksymalne dopuszczalne narzuty próbkowania Circuit lub `None` bez maksimum. | None / liczba całkowita > 1 | 100 |

W poniższym przykładzie ustawienie poziomu mitigacji na 1 początkowo wyłącza mitigację ZNE, ale ustawienie `zne_mitigation` na `True` nadpisuje odpowiednią konfigurację z `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Dane wyjściowe
Wyjście Circuit function to [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), który zawiera dwa pola:

- Jeden lub więcej obiektów [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Można się do nich odwoływać bezpośrednio z `PrimitiveResult`.

- Metadane na poziomie zadania.

Każdy `PubResult` zawiera pole `data` i `metadata`.

- Pole `data` zawiera co najmniej tablicę wartości oczekiwanych (`PubResult.data.evs`) i tablicę błędów standardowych (`PubResult.data.stds`). Może też zawierać więcej danych, w zależności od użytych opcji.

- Pole `metadata` zawiera metadane na poziomie PUB (`PubResult.metadata`).

Poniższy fragment kodu opisuje format `PrimitiveResult` (i powiązanego `PubResult`).